# Engineering of Data Analysis - lecture 2 - 2

Engineering of Data Analysis is an elective course of the MSc in Business Analytics of NOVA SBE. The course assumes knowledge of Python programming language and Pandas framework.

This notebook presents examples and exercises of lecture 2.

[//]: # (We will be using latex for fomulas)

<script type="text/javascript"
        src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.0/MathJax.js?config=TeX-AMS_CHTML"></script>


## Setup

We will start by download the datasets to be used in this notebook. Alternatively, in Colab, you could upload the dataset.

In [ ]:
!pip install gdown
!gdown 'https://drive.google.com/uc?id=1fUqkrX9svaDx_H3IH00eDWMfA0rRffGD'
!unzip -o eoda2425-lec2-data.zip

## Programming (with Pandas)

We now show how to program in Python adopting a programming paradigm that consists in applying transformations to Dataframes.

We will use the popular [**Pandas**](https://pandas.pydata.org/) library for our examples, althouth the underlying paradigm can be found in different libraries and frameworks.

For using Pandas, we start by importing *pandas*.

In [ ]:
# imports pandas
import pandas as pd


### Data model : DataFrame

In *Pandas*, a table is represented as a [**DataFrame**](https://pandas.pydata.org/docs/reference/frame.html). (follow the link for DataFrame documentation)

There are multiple ways to create an initial DataFrame. For example, you can create date from a Python dictionary, as follows:

In [ ]:
population = pd.DataFrame( { "country": ["PT", "ES", "DE"] , \
                            "population": [10276617, 46937060, 83019213]})

print( population)


Pandas will maintain an additional column, the index, with a increasing integer. This column - the first column when printing the dataframe - is created automatically.

#### Loading DataFrame from CSV files

More often, will want to load the data from files. To create a DataFrame from a CSV file, you can use the ```load_csv``` function.

Note: If the following code fails, the most likely reason is that you do not have the *data* directory with the data files.

In [ ]:
import os

# Let's create a PATH in a OS independent way
# File lec2-example.csv is in directory data
fileName = os.path.join( "data", "lec2-example.csv")

# Read a CSV file into a DataFrame
df = pd.read_csv(fileName)

print( df)


#### Saving DataFrame into CSV files

You can save a DataFrame into a CSV file using ```to_csv``` function.

In [ ]:
import os

# Let's create a PATH in a OS independent way
# File lec2-saved.csv will be in directory data
fileName = os.path.join( "data", "lec2-saved.csv")

# Save DataFrameRead a CSV file into a DataFrame
df.to_csv( fileName)


Please check the file created. Is it the same as the original lec2-saved.csv?

No, it has an additional column with the row number. You can also save the DataFrame without this number by using the ```index=False``` option.

In [ ]:
import os

# Let's create a PATH in a OS independent way
# File lec2-saved-noindex.csv will be in directory data
fileName = os.path.join( "data", "lec2-saved-noindex.csv")

# Save DataFrameRead a CSV file into a DataFrame
df.to_csv( fileName, index=False)


### Data processing with Pandas

We now show the transformations necessary to perform the exercises proposed above.


#### Selecting rows based on conditions (map transformation)

It is possible to select the rows for which a column has a given value as follows:

In [ ]:
# Select the persons that are good company.
good = df[df["Company"]=="Good"]

print(good)

In [ ]:
# Select the persons that are good company and have educational level larger than 3.
goodEd3plus = df[(df["Company"]=="Good") & (df["Educational level"]>=3.0)]

print(goodEd3plus)

#### Selecting a subset of the columns (map transformation)

Often, we do not need all data that is in a table. We can get rid of the data we do not need by selecting the columns we want using the following syntax ```dataframe[[col1,col2,...]]```.

In the following example we create a new DataFrame containing only the Name and Age columns.

In [ ]:
# Select the persons that are good company.
person_age = df[["Name","Age"]]

print(person_age)

#### Applying reduce/aggregation functions

Pandas allow to compute the reduction/aggregation for the values of one or multiple columns.

You must select the columns for which you want to perform the computation, and then call the reduce/aggregation function.

The following example computes first, the minimum age (```min```function), and then the minimum of both *Age* and *Educational level* at the same time. Pandas has multiple useful aggregation functions, including, maximum (```max```), minimum (```min```), mean (```mean```), median (```median```), standard deviation (```std```), etc. - check the [**DataFrame** documentation](https://pandas.pydata.org/docs/reference/frame.html) for the list of available functions.

In [ ]:
minAge = good["Age"].min()
print( "Minimum age is ")
print( minAge)

mins = good[["Age","Educational level"]].min()
print( "Minimum information for several columns now")
print( mins)


It is also possible to apply different aggregation function to different columns using the [```dataframe.agg()``` function.](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html)


In [ ]:
stats = good.agg({'Age' : 'min', 'Educational level' : 'max'})
print( "Minimum and max in the same operation")
print( stats)

Wait, this was not what we wanted in the first place - we want the information about the youngest person that is a good company.

Function ```nsmallest(num elems, columns)``` allow to compute that.

In [ ]:
youngestGood = good.nsmallest(1,["Age"])
print( good.nsmallest(1,["Age"]))


#### Series

In Pandas, a Series is a sequence of values of any type, with an associated index - each value has an associated index.

A Series can be created from a Dataframe by using the syntax ```dataframe[col]```.


In [ ]:
#Creates a series with the values of column Name
good["Name"]

Some Dataframe functions also generate series as a result. For example, the result of an aggregation over multiple columns in a Series.


In [ ]:
type(good[["Age","Educational level"]].min())

A Series can be converted into a Dataframe using the ```to_frame``` function

In [ ]:
good["Name"].to_frame()

#### Applying reduce/aggregation functions per group

```groupby([cols])``` allows to group elements of a DataFrame before applying an aggregation function to each of the groups.

The following example computes the lowest age for each value of Company.


In [ ]:
youngest = df[["Age","Company"]].groupby(["Company"]).min()
print( youngest)

youngestAny = youngest.idxmin()
print( "The youngest person is " + youngestAny["Age"])


What if we wated to get the full row for the youngest person in each group. In this case, nsmallest cannot be applied to a *groupby*. To address this, we can define a function that is called for each group.

Note: `include_groups=False` requires Pandas ≥ 2.2. On older versions, omit this parameter — the grouping column will be included in the function's input.


In [ ]:
def mysmallest( p):
    return p.nsmallest(1,["Age"])

stats = df.groupby("Company").apply(mysmallest,include_groups=False)
print( stats)

For simple functions, instead of defining a function using the normal syntax, it is possible to use a lambda function. The lambda function is definded with the following syntax:

```lambda arguments : expression```

This is equivalent to define the function

```
def myfun( arguments):
    return expression
```


In [ ]:
stats = df.groupby("Company").apply(lambda p: p.nsmallest(1,["Age"]),include_groups=False)
print( stats)

### Code for exercises above

In [ ]:
# what is the average age for each type of company?


In [ ]:
# Know which group has lower average Education level: good or bad companies ?




## Functions over multiple Dataframe

Often, data will be in multiple tables/Dataframes. To process data, it is necessary to execute operations over these tables.

We now introduce some of the operations available on Pandas for combining multiple tables.


### Appending tables

Sometimes, we have data over which we want to perform a computation that is in two different Dataframes - e.g. because we read data from different files.

Consider we have the following tables:

| country | population |
|---------|------------|
| PT | 10276617 |
| ES | 46937060 |
| DE | 83019213 |

and

| capital | population | country |
|---------|------------|---------|
| Brasilia | 211049519 | BR |
| Mexico City | 127575529 | MX |
| Montevideu | 3461731 | UY |

```pd.concat([dataframe,dataframe2])``` creates a new table that combines the values in the first dataframe followed by the values in dataframe2, using the columns name. If some column does not exist in one table, the value of the rows will be **NaN**.

| country | population | capital |
|---------|------------|---------|
| PT | 10276617 | NaN |
| ES | 46937060 | NaN |
| DE | 83019213 | NaN |
| BR | 211049519 | Brasilia |
| MX | 127575529 | Mexico City |
| UY | 3461731 | Montevideu |

| country | population | language |
|---------|------------|---------|
| PT | 10276617 | Portuguese |
| ES | 46937060 | Spanish |
| ES | 46937060 | Catalan |
| DE | 83019213 | German |
| BR | 211049519 | Portuguese |
| MX | 127575529 | Spanish |
| UY | 3461731 | NaN |
| AR | NaN | Spanish |
| IT | NaN | Italian |

The following code show the example running.


In [ ]:
population1 = pd.DataFrame( { "country": ["PT", "ES", "DE"] , \
                            "population": [10276617, 46937060, 83019213]})

print( population1)



In [ ]:
population2 = pd.DataFrame( {"capital" : ["Brasilia", "Mexico City", "Montevideu"],\
                            "population": [211049519, 127575529, 3461731], \
                            "country": ["BR", "MX", "UY"]})

print( population2)



In [ ]:
population = pd.concat([population1,population2])
print( population)

### Joining tables

More interestingly, we might want to combine the columns from one or more tables into a new table.

Consider that we have the following two tables. The first table has a list of countries and their population.

| country | population |
|---------|------------|
| PT | 10276617 |
| ES | 46937060 |
| DE | 83019213 |

The second table has the language spoken in each country.

| country | language |
|---------|----------|
| PT | Portuguese |
| ES | Spanish |
| MX | Spanish |
| AR | Spanish |
| DE | German |
| IT | Italian |
| BR | Portuguese |


| country | language |
|---------|----------|
| DE | German |
| PT | Portuguese |
| ES | Spanish |



If we want to compute the number of persons that speak each language, it would be interesting to have a single table with the country, population and language columns. To this end, we need to combine both of the previous tables (this can also be seen as extending the first table with the values of the second table).

What we want to achieve is the following table, with columns country, population and language:

| country | population | language |
|---------|------------|----------|
| PT | 10276617 | Portuguese |
| ES | 46937060 | Spanish |
| DE | 83019213 | German |


The ```dataframe.join(dataframe2,on=column,how="left"|"right"|"inner")``` function allows to combine two tables. By default, the two table are combined using the index, i.e., a row with index **i** in dataframe is combined with the row with index **i** in dataframe2.

The ```dataframe.merge(dataframe2,left_on=column,right_on=column,how="left"|"right"|"inner")``` function does the same as join, but allows to specify the columns to be used to combine in both dataframes.

In our example, we want to combine the row with country value **N** of the first table with the row with country value **N** in the second table. To this end, we can use the ```on="country"```to specify that we want to use the value of the column *country* in the first table. For dataframe2, the index will be used -- this require us to change the index of dataframe2, using the ```dataframe2.set_index( col)``` function.

This example is coded in the following cells.

In [ ]:
language = pd.DataFrame( { "country0": ["PT", "ES", "MX", "AR", "DE", "BR"] , \
                            "language": ["Portuguese", "Spanish", "Spanish", "Spanish", "German", "Portuguese"]})

print( language)



In [ ]:
countries1 = population1.join(language.set_index("country0"),on="country")

print(countries1)

In [ ]:
countries1merge = population1.merge(language,left_on="country",right_on="country0")

print(countries1merge)

Based on the population Dataframe computed before, compute the number of persons that speak each language.

In [ ]:
## TODO Complete



### Join type : left (default)

The way join works varies depending on the

In a left join, each row of *dataframe* is combined with all possible values of *dataframe2*. If no row in the second dataframe matches the joining column of the first, then the value for the columns will be **NaN**.

For better exemplifying, we start by extending our language table to include one other language for Spain : Catalan.

| country | language |
|---------|----------|
| PT | Portuguese |
| ES | Spanish |
| ES | Catalan |
| MX | Spanish |
| AR | Spanish |
| DE | German |
| IT | Italian |
| BR | Portuguese |


In [ ]:
languageExt = pd.DataFrame( { "country": ["PT", "ES", "ES", "MX", "AR", "DE", "IT", "BR"] , \
                            "language": ["Portuguese", "Spanish", "Catalan", "Spanish", "Spanish", "German", "Italian", "Portuguese"]})

print( languageExt)


In [ ]:
countries = population[["country","population"]].join(languageExt.set_index("country"),on="country")

print(countries)

In [ ]:
# merge with just "on" will use the column with the specified name in both Dataframes
countries = population[["country","population"]].merge(languageExt,on="country")

print(countries)

The line for Uruguay (UY) has **NaN** in the language column.


### Join type : right

In a right join, each row of *dataframe2* is combined with all possible values in the first *dataframe*. If no row in the first dataframe matches the joining column of the second, then the value for the columns will be **NaN**.



In [ ]:
countries = population.join(languageExt.set_index("country"),on="country",how="right")

print(countries)


In [ ]:
countriesMerge = population.merge(languageExt,on="country")

print(countriesMerge)


### Join type : inner

In an inner join, each row of *dataframe* is combined with all possible values in the second *dataframe*. If no row in the second dataframe matches the joining column of the first, then the row will not be part of the result.



In [ ]:
countries = population.join(languageExt.set_index("country"),on="country",how="inner")

print(countries)


In [ ]:
countries = population.merge(languageExt,on="country",how="inner")

print(countries)


### Join type : outer

In an outnner join, each row of *dataframe* is combined with all possible values in the second *dataframe*. If no row exists in any of the dataframes, both row will appear in the final result.



In [ ]:
countries = population.join(languageExt.set_index("country"),on="country",how="outer")

print(countries)

In [ ]:
countries = population.merge(languageExt,on="country",how="outer")

print(countries)

## Exercises


### Exercise 1

Based on the given data, compute the population that speaks Spanish.

### Exercise 2

For each language for which there is some known population, compute the population that speaks such language.